In [1]:

import numpy as np
import tensorflow as tf
import csv
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers
from scipy.optimize import minimize

In [2]:
data = []

with open('new_training_data.csv', newline='') as csvfile:
    reader = csv.reader(csvfile)
    next(reader)
    for row in reader:
        float_row = [float(item) for item in row[1:]]
        data.append(float_row)

data = np.array(data)
print(data[0,0:6],data[0,7])


[4.e-01 2.e+02 2.e+02 5.e-01 1.e-01 7.e-01] 97.85050016687262


In [3]:
param_bounds = [
    (0, 1),      # IR
    (50, 500),   # NG
    (50, 500),   # PS
    (0.1, 1),      # PC
    (0.1, 1),      # PM
    (0.3, 0.75)      # NMP
]

def normalize_params(X, bounds):
    X_norm = np.empty_like(X)
    for i, (min_val, max_val) in enumerate(bounds):
        X_norm[:, i] = (X[:, i] - min_val) / (max_val - min_val)
    return X_norm

def denormalize_params(X_norm, bounds):
    X = np.empty_like(X_norm)
    for i, (min_val, max_val) in enumerate(bounds):
        X[:, i] = X_norm[:, i] * (max_val - min_val) + min_val
    return X

In [4]:

X = data[:, 0:6]  # Hyperparameters
y = data[:, 7:]  # Results

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
X_normalized = normalize_params(X, param_bounds)
print(X_normalized)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1))
def buid_model():
    model = keras.Sequential([
        layers.Input(shape=(1,)),      
        layers.Dense(64, activation='relu'),  
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(6, activation='sigmoid')  # 6 outputs = hyperparameters (normalized 0–1)
    ])


    model.compile(optimizer='adam', loss='mse')
    return model
model = buid_model()    

[[0.4        0.33333333 0.33333333 0.44444444 0.         0.88888889]
 [0.4        0.33333333 0.33333333 0.44444444 0.05555556 0.88888889]
 [0.4        0.33333333 0.33333333 0.44444444 0.22222222 0.88888889]
 ...
 [0.9        0.71111111 1.         0.77777778 0.         0.88888889]
 [0.9        0.71111111 1.         0.77777778 0.05555556 0.88888889]
 [0.9        0.71111111 1.         0.77777778 0.22222222 0.88888889]]


In [5]:
model.fit(y_scaled, X_normalized)

8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0864


In [6]:
import jpype
import jpype.imports

def postprocess_params(params):
    # Ensure it's a clean copy and 1D
    params = params.flatten().copy()

    # Process each parameter individually
    ps = int(round(params[2]))
    ps = max(50, min(ps, 500))
    if ps % 2 != 0:
        ps += 1 if ps < 500 else -1

    nmp = float(round(params[5]))
    nmp = max(0.3, min(nmp, 0.75))

    ng = int(round(params[1]))
    ng = max(50, min(ng, 500))

    ir = float(np.clip(params[0], 0, 1))
    pc = float(np.clip(params[3], 0, 1))
    pm = float(np.clip(params[4], 0, 1))

    numeric_params = np.array([ir, ng, ps, pc, pm, nmp])

    # String version for Java input
    java_params = [
        f"{ir:.6f}",  # float
        str(ng),      # int
        str(ps),      # int
        f"{pc:.6f}",  # float
        f"{pm:.6f}",
          f"{nmp:.6f}"  # float
    ]

    return numeric_params, java_params

def run_java_algorithm(params):
    if not jpype.isJVMStarted():
        jpype.startJVM(classpath=["C:/Users/USER/Desktop/my_projects/optimization_with_java/bin"])
    GGA = jpype.JClass("GGA.GGA")  # Just the class name
    java_params = jpype.JArray(jpype.JString)([str(p) for p in params])
    print("→ Running Java GGA with params:", java_params)
    return GGA.tryData(java_params)

def objective(scaled_result_input):
    # Reshape input (1 feature)
    scaled_result_input = np.array(scaled_result_input).reshape(1, -1)
    
    # Predict normalized hyperparameters
    predicted_params_norm = model.predict(scaled_result_input, verbose=0)

    # Denormalize to real parameter space
    predicted_params = denormalize_params(predicted_params_norm, param_bounds)

    # Optionally postprocess (round integers, etc.)
    predicted_params ,j= postprocess_params(predicted_params)

    # Evaluate GGA algorithm on MKP using these predicted parameters
     # Get numeric params only
    score = run_java_algorithm(j)  # You define this

    return -score  # Negate if you want to maximize


# Start from mean result (scaled)
initial_result = 1#scaler_y.transform(np.mean(y).reshape(1, -1))[0]

# Bounds for input result ∈ scaled space (roughly [-2, 2] for standardized data)
bounds = [(-2, 2)]  # only one input dimension now

# Run optimization
result = minimize(objective, initial_result, bounds=bounds, method='L-BFGS-B',options={
        'maxiter': 5,
        'disp': True,       # Show optimization info
        'gtol': 1e-6,        # Stop if gradient norm < 1e-6
    })


→ Running Java GGA with params: ['0.526488', '282', '288', '0.584142', '0.511277', '0.750000']
→ Running Java GGA with params: ['0.526488', '282', '288', '0.584142', '0.511277', '0.750000']


In [7]:
best_result=-result.fun
# Best result input (scaled)
best_result_scaled = scaler_y.transform(np.array([[best_result]]))
# Predict best hyperparameters from it
best_params_norm = model.predict(best_result_scaled, verbose=0)
best_params = denormalize_params(best_params_norm, param_bounds)
best_params = postprocess_params(best_params)
print(result)


print("✅ Best parameters found (original scale):", best_params)
print("initial_result", initial_result)
print("🎯 best result", best_result)

  message: CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
  success: True
   status: 0
      fun: -99.0
        x: [ 1.000e+00]
      nit: 0
      jac: [ 0.000e+00]
     nfev: 2
     njev: 1
 hess_inv: <1x1 LbfgsInvHessProduct with dtype=float64>
✅ Best parameters found (original scale): (array([  0.5734598 , 294.        , 308.        ,   0.64147353,
         0.4514541 ,   0.75      ]), ['0.573460', '294', '308', '0.641474', '0.451454', '0.750000'])
initial_result 1
🎯 best result 99.0


In [8]:
import numpy as np
import subprocess
import csv
from scipy.optimize import minimize


def read_new_training_data(filepath='new_training_data02.csv'):
    new_data = []
    with open(filepath, newline='') as csvfile:
        reader = csv.reader(csvfile)
        next(reader)
        for row in reader:
            float_row = [float(item) for item in row[1:]]
            new_data.append(float_row)
    return np.array(new_data)

def retrain_data(params):
    if not jpype.isJVMStarted():
        jpype.startJVM(classpath=["C:/Users/USER/Desktop/my_projects/optimization_with_java/bin"])
    GGA = jpype.JClass("GGA.GGA")  # Just the class name
    java_params = jpype.JArray(jpype.JString)([str(p) for p in params])
    print("→ Running Java GGA with params:", java_params)
    return GGA.retrainingData(java_params)
def retrain_model(model, scaler_y, new_data):
    X_new = new_data[:, 0:6]
    y_new = new_data[:, 7:]
    X_normalized = normalize_params(X_new, param_bounds)
    y_new_scaled = scaler_y.transform(y_new.reshape(-1, 1))
    
    model.fit(
    y_new_scaled,
    X_normalized,
    epochs=100,               # You can tune this
    validation_split=0.2,
    verbose=0                 # Or 1 to see training output
)

def optimize_params():
    initial_result = scaler_y.transform(np.mean(y).reshape(1, -1))[0]
    bounds = [(-2, 2)] 
    result = minimize(objective, initial_result, bounds=bounds, method='L-BFGS-B')
    return result.x.reshape(1, -1)


# ==== Main Retraining Loop ====
##model = buid_model()  

average_result = retrain_data(best_params[1])
max_average = best_result  # better start with very low
num_iterations = 10  # Number of retrain cycles
for iteration in range(num_iterations):
    print(f"\n--- Iteration {iteration + 1} ---")
     # Step 1: Read new training data from Java output or other source
    new_training_data = read_new_training_data()
    X_new = new_training_data[:, 0:6]
    y_new = new_training_data[:, 7:]
    new_scaler_y= StandardScaler()
    new_scaled_y = new_scaler_y.fit_transform(y_new.reshape(-1, 1))
    print("→ New training data shape:", new_training_data.shape)
    # Step 2: Recreate the model & retrain your model with new data
    
    retrain_model(model,  new_scaler_y, new_training_data)

    # Step 3: predict new best parameters based on the current model
    scaled_best_result = new_scaler_y.transform(np.array([[99.80]]))
    new_params_norm = model.predict(scaled_best_result, verbose=0)
    new_params = denormalize_params(new_params_norm, param_bounds)

    # Step 4: Postprocess (Java & numeric)
    numeric_params, java_params = postprocess_params(new_params)
    print("→ Optimized & Postprocessed Params:", java_params)


    # Step 5: Run Java algorithm with these parameters & Calculate average result assuming it is in the last column
    average_result = retrain_data(java_params)
    print(f"→ Average Result in Training Data: {average_result:.4f}")

    # Track the best parameters if the current average is better
    if average_result > max_average:
        max_average = average_result
        best_params = numeric_params.copy()  # copy to avoid reference issues
        print(f"→ New Maximum Result Found: {max_average:.4f}")


    print("→ Generated Parameters After Retraining:")
    print(f"   IR  = {numeric_params[0]:.3f}   NG  = {int(numeric_params[1])} PS  = {int(numeric_params[2])} (even)PC  = {numeric_params[3]:.3f} PM  = {numeric_params[4]:.3f} NMP = {int(numeric_params[5])}")

print("\nBest possible parameters found during iterations:",best_params)


→ Running Java GGA with params: ['0.573460', '294', '308', '0.641474', '0.451454', '0.750000']

--- Iteration 1 ---
→ New training data shape: (10, 8)
→ Optimized & Postprocessed Params: ['0.590075', '301', '320', '0.671745', '0.416814', '0.750000']
→ Running Java GGA with params: ['0.590075', '301', '320', '0.671745', '0.416814', '0.750000']
→ Average Result in Training Data: 99.0000
→ Generated Parameters After Retraining:
   IR  = 0.590   NG  = 301 PS  = 320 (even)PC  = 0.672 PM  = 0.417 NMP = 0

--- Iteration 2 ---
→ New training data shape: (10, 8)
→ Optimized & Postprocessed Params: ['0.635993', '327', '358', '0.763300', '0.308274', '0.750000']
→ Running Java GGA with params: ['0.635993', '327', '358', '0.763300', '0.308274', '0.750000']
→ Average Result in Training Data: 99.0000
→ Generated Parameters After Retraining:
   IR  = 0.636   NG  = 327 PS  = 358 (even)PC  = 0.763 PM  = 0.308 NMP = 0

--- Iteration 3 ---
→ New training data shape: (10, 8)
→ Optimized & Postprocessed Par